# Appendix: mixtures of multichannel hidden Markov models

Section 4 extends every clustering result to hidden Markov models with multichannel
observations, and is the only part of the paper with no figure. This notebook gives it one,
by running the whole numerical section again on that mechanism and asking whether the
conclusions survive.

**What changes.** A component is now a homogeneous HMM: a latent chain on four hidden
states drives five conditionally independent channels of five letters each. A sequence is an
$(n, 5)$ array rather than an $(n,)$ one, and the dissimilarity is the multichannel OM of
Definition 4.1, $c_{\mathrm{sub}} = \sum_c c_{\mathrm{sub}}^{(c)}$ and
$\delta = \sum_c \delta^{(c)}$, with constant per-channel costs. That aggregation gives
$M^{\mathrm{mc}} = 10$ and $\delta^{\mathrm{mc}} = 5$ against 2 and 1 in the univariate
case, and Assumption 1 holds — checked channel by channel, since the product alphabet has
$5^5 = 3125$ letters and the triangle inequality on it is a 227 GiB table.

The costs are fixed in advance rather than estimated from the data, unlike the TRATE path.
$\Gamma^{(n)}$ is estimated on an independent sample, and a cost scheme derived from that
sample would make the dissimilarity depend on which sequences happened to be drawn.

**What does not change.** Everything downstream of the dissimilarity matrix. The same
estimator of $\Gamma^{(n)}$, the same simultaneous intervals, the same four clustering
algorithms, the same rules for $K$ — none of them knows which mechanism produced the matrix
it reads.

**The grid axis.** HMMs have no Dirichlet concentration, so $\alpha$ is read as the emission
concentration $\alpha_B$, with the same meaning: small values give sharply peaked emissions
and components that are easy to tell apart.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from experiments import wilson_interval
from figures import DIVERGING_CMAP, PAPER_STYLE, heatmap, plot_paths, save

## Setup

`run_hmm_night.sh` produces the tables: the paths, the grid, and the rules for $K$, in that
order, about ten hours in all. The Markov tables are loaded alongside so every figure can be
read against its univariate counterpart.

In [ ]:
RECOMPUTE = False        # True re-runs the three HMM sweeps: about 10 h on 32 cores

RESULTS = Path("results")
FIGURES = Path("Figures/HMM")
HORIZON = 1000
ALPHAS  = [0.1, 0.2, 0.3, 0.4, 0.5, 1.0, 5.0, 10.0]
KS      = list(range(2, 11))
ALGOS   = ["average", "pam"]
ALGOS_ALL = ["single", "complete", "average", "pam"]

if RECOMPUTE:
    import subprocess
    subprocess.run(["./run_hmm_night.sh"], check=True)


def load(stem, mechanism):
    """Read a table, or None if the sweep that writes it has not reached it yet.

    The three HMM sweeps run in sequence over about ten hours, so the notebook has to be
    readable while they are still going: a missing table skips its section rather than
    stopping the run.
    """
    suffix = ("_hmm" if mechanism == "hmm"
              else "_main" if ("recovery" in stem or "path" in stem) else "_final")
    path = RESULTS / f"{stem}{suffix}.csv"
    if not path.exists():
        print(f"  {path.name}: not written yet, its section will be skipped")
        return None
    return pd.read_csv(path)


def complete_cells(df):
    """Keep only cells every mixture has finished, so no cell is averaged over fewer draws."""
    if df is None:
        return None
    counts = df.groupby(["alpha", "K"]).mixture_id.nunique()
    full = counts[counts == counts.max()].index
    return df[df.set_index(["alpha", "K"]).index.isin(full)]


hmm_cluster = load("recovery_cluster", "hmm")
hmm_eta     = load("recovery_eta", "hmm")
hmm_khat    = load("khat_grid", "hmm")
hmm_path    = load("path_cluster", "hmm")

mk_cluster  = load("recovery_cluster", "markov")
mk_eta      = load("recovery_eta", "markov")
mk_khat     = load("khat_grid", "markov")
mk_path     = load("path_cluster", "markov")

def at_horizon(df):
    return None if df is None else complete_cells(df[df.n == HORIZON])


hmm_cluster, mk_cluster = at_horizon(hmm_cluster), at_horizon(mk_cluster)
hmm_eta, mk_eta = at_horizon(hmm_eta), at_horizon(mk_eta)
hmm_khat, mk_khat = at_horizon(hmm_khat), at_horizon(mk_khat)

# The HMM sweep is the shorter of the two while it is still running, so the grids are drawn
# on the alphas it has finished; the Markov panel is restricted to the same rows, otherwise
# the two heatmaps would not be comparable line by line.
ALPHAS_PRESENT = sorted(hmm_cluster.alpha.unique()) if hmm_cluster is not None else ALPHAS

for label, df in (("HMM", hmm_cluster), ("Markov", mk_cluster)):
    if df is None:
        continue
    cells = df.groupby(["alpha", "K"]).ngroups
    print(f"{label:<7}: {df.mixture_id.nunique()} mixtures per cell over {cells} complete "
          f"cells, alphabet {int(df.d.iloc[0])}, N = {int(df.N.iloc[0])}")
print(f"grids drawn on alpha in {ALPHAS_PRESENT}")

for name in ("mk_cluster", "mk_eta", "mk_khat"):
    df = globals()[name]
    if df is not None:
        globals()[name] = df[df.alpha.isin(ALPHAS_PRESENT)]

## Reading a cell

In [ ]:
def grid(df, value, agg="mean"):
    table = df.pivot_table(index="alpha", columns="K", values=value, aggfunc=agg)
    return table.reindex(index=ALPHAS_PRESENT, columns=KS).to_numpy(dtype=float)


def verdict_balance(eta):
    """Pr(separated) - Pr(nonseparated): +1 all separated, -1 all not, 0 no verdict."""
    e = eta.assign(sep=(eta.separation_status == "separated").astype(int),
                   non=(eta.separation_status == "nonseparated").astype(int))
    return grid(e, "sep") - grid(e, "non"), grid(e, "sep"), grid(e, "eta_hat", agg="median")

## Plausibility of the separation condition

The two mechanisms side by side. Blue where $\eta_n > 0$ is established at 95% simultaneous
confidence, red where $\eta_n < 0$ is, pale where the interval cannot decide.

In [ ]:
with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), sharey=True)
    for ax, (label, eta) in zip(axes, (("Markov chains", mk_eta),
                                       ("multichannel HMMs", hmm_eta))):
        balance, _, margin = verdict_balance(eta)
        im = heatmap(ax, balance, ALPHAS_PRESENT, KS, label, annot=margin, cmap=DIVERGING_CMAP,
                     vmin=-1.0, vmax=1.0)
    cb = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, ticks=[-1, 0, 1])
    cb.ax.set_yticklabels(["all not\nseparated", "no\nverdict", "all\nseparated"], fontsize=6)
    cb.outline.set_visible(False)
    save(fig, FIGURES, "separation_hmm_vs_markov")
    plt.show()

## Recovery at known $K$

Average linkage and PAM, on the same grid.

In [ ]:
with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.0), sharey=True)
    for ax, algo in zip(axes, ALGOS):
        sub = hmm_cluster[hmm_cluster.algorithm == algo]
        im = heatmap(ax, grid(sub, "exact_recovery"), ALPHAS_PRESENT, KS, algo,
                     annot=grid(sub, "ari"))
    cb = fig.colorbar(im, ax=axes, fraction=0.03, pad=0.02)
    cb.set_label("probability of exact recovery (annotated: mean ARI)", fontsize=8)
    cb.outline.set_visible(False)
    save(fig, FIGURES, "recovery_hmm")
    plt.show()

## Selecting $K$

The safeguarded rule against the applied default, on HMM dissimilarities.

In [ ]:
if hmm_khat is None:
    print("the K-selection sweep has not written its table yet; skipping this figure")
RULES = ["safeguard", "asw-pam", "stated"]
TITLES = {"safeguard": "safeguarded profile rule",
          "asw-pam": "maximal silhouette width (PAM)",
          "stated": r"threshold of Theorem 3.9"}

if hmm_khat is not None:
  with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.6), sharey=True)
    for ax, rule in zip(axes, RULES):
        sub = hmm_khat[hmm_khat.rule == rule]
        im = heatmap(ax, grid(sub, "k_correct"), ALPHAS_PRESENT, KS, TITLES[rule],
                     annot=grid(sub, "exact_recovery"))
    cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
    cb.set_label(r"$\Pr(\hat K = K)$ (annotated: $\Pr$ exact partition)", fontsize=8)
    cb.outline.set_visible(False)
    save(fig, FIGURES, "k_selection_hmm")
    plt.show()

## Recovery against the horizon

In [ ]:
COL = {"single": "#1b6ca8", "average": "#6a51a3", "pam": "#c0392b"}
NAMES = {"single": "single linkage", "average": "average linkage", "pam": "PAM"}
HZ = np.sort(hmm_path.n.unique())


def curves(df, algorithm, value="ari"):
    sub = df[df.algorithm == algorithm]
    reps = sub.pivot_table(index=["mixture_id", "dataset_id"], columns="n", values=value)
    means = sub.pivot_table(index="mixture_id", columns="n", values=value)
    return reps.reindex(columns=HZ).to_numpy(), means.reindex(columns=HZ).to_numpy()


for algo in ALGOS:
    reps, means = curves(hmm_path, algo)
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(figsize=(5.2, 4.0))
        plot_paths(ax, HZ, reps, means, COL[algo], f"ARI, {NAMES[algo]}",
                   rf"{NAMES[algo]} --- HMM, $N = {int(hmm_path.N.iloc[0])}$, "
                   rf"$K = {int(hmm_path.K.iloc[0])}$")
        save(fig, FIGURES, f"ari_path_{algo}_hmm")
        plt.show()

## Do the conclusions survive?

The question the appendix exists to answer. Every headline number of the section, on both
mechanisms, over the same grid.

In [ ]:
def headline(cluster, eta, khat, label):
    row = {"mechanism": label}
    for algo in ALGOS:
        s = cluster[cluster.algorithm == algo]
        row[f"exact, {algo}"] = f"{100 * s.exact_recovery.mean():.1f}%"
    if khat is not None:
        for rule in ("safeguard", "asw-pam", "stated"):
            s = khat[khat.rule == rule]
            row[f"K_hat, {rule}"] = f"{100 * s.k_correct.mean():.1f}%"
    row["separated"] = f"{100 * (eta.separation_status == 'separated').mean():.0f}%"
    return row


print(pd.DataFrame([headline(mk_cluster, mk_eta, mk_khat, "Markov"),
                    headline(hmm_cluster, hmm_eta, hmm_khat, "HMM")]).to_string(index=False))

print("\nrecovery against the finite-horizon geometry, both mechanisms")
rows = []
edges  = [-np.inf, 0.0, 0.05, 0.20, np.inf]
labels = ["eta<0", "0-.05", ".05-.20", ">.20"]
for label, cluster, eta in (("Markov", mk_cluster, mk_eta), ("HMM", hmm_cluster, hmm_eta)):
    m = cluster.merge(eta[["alpha", "K", "mixture_id", "eta_hat"]],
                      on=["alpha", "K", "mixture_id"], how="left")
    m["band"] = pd.cut(m.eta_hat, edges, labels=labels)
    for band in labels:
        s = m[m.band == band]
        if not len(s):
            continue
        row = {"mechanism": label, "eta band": band, "mixtures": len(s) // len(ALGOS_ALL)}
        for a in ALGOS:
            row[a] = f"{100 * s[s.algorithm == a].exact_recovery.mean():.0f}%"
        rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

The chain the section rests on is the thing to check, not the absolute rates: the two
mechanisms put different amounts of information in a sequence, so $\eta_n$ lives on
different scales and the grids are not directly comparable cell by cell. What has to carry
over is the relation — recovery governed by $\eta_n$, the safeguarded rule ahead of the
silhouette, the stated threshold returning $\hat K = 1$ — and it is the last table that says
whether it does.